# Causal-aware early warning: Step 1 (data, labels, baseline models)
Expected layout in Google Drive: `MyDrive/Dataset/` with the five subfolders you downloaded.

In [ ]:
!pip install -q xlrd
import pandas as pd, numpy as np
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

BASE = '/content/drive/MyDrive/My Work/Research/AI+ BBA/Supply Chain_GNN/Dataset'   # change if your folder is elsewhere
import os; print(os.listdir(BASE))

['5. NY Fed', '3. Brent and WTI spot prices from EIA', '2. Enso', '1. IMF Hormuz']


## 1. Load the files

In [ ]:
enso = pd.read_csv(f"{BASE}/2. Enso/dataset_enso_2026.csv", encoding="latin-1")
enso.columns = [c.strip() for c in enso.columns]
enso["month"] = pd.to_datetime(enso["Date"], format="%m/%d/%Y").dt.to_period("M")
enso = enso[["month", "ONI", "Nino 3.4 SST Anomaly"]].rename(
    columns={"ONI": "oni", "Nino 3.4 SST Anomaly": "nino34_anom"})

brent = pd.read_csv(f"{BASE}/3. Brent and WTI spot prices from EIA/brent-monthly.csv")
brent["month"] = pd.to_datetime(brent["Date"]).dt.to_period("M")
brent = brent[["month", "Price"]].rename(columns={"Price": "brent"})

g = pd.read_excel(f"{BASE}/5. NY Fed/gscpi_data.xls", sheet_name="GSCPI Monthly Data")
g = g.iloc[:, :2]; g.columns = ["date", "gscpi"]
g = g.dropna(subset=["date", "gscpi"])
g["month"] = pd.to_datetime(g["date"], format="%d-%b-%Y").dt.to_period("M")
gscpi = g[["month", "gscpi"]]

tr = pd.read_csv(f"{BASE}/1. IMF Hormuz/straits-transits.csv", parse_dates=["date"])
tr["month"] = tr["date"].dt.to_period("M")
hormuz = tr.groupby("month").agg(hz_transits=("n_total", "mean"),
                                 hz_days=("n_total", "size")).reset_index()

## 2. Merge into one monthly panel

In [ ]:
df = (gscpi.merge(brent, on="month", how="outer")
           .merge(enso, on="month", how="outer")
           .merge(hormuz, on="month", how="left")
           .sort_values("month").reset_index(drop=True))
df = df[(df.month >= "1998-01") & (df.month <= "2026-09")]

## 3. Features (only information known at month t)

In [ ]:
df["brent_ret1"] = df["brent"].pct_change(1)
df["brent_ret3"] = df["brent"].pct_change(3)
df["brent_z12"] = (df["brent"] - df["brent"].rolling(12).mean()) / df["brent"].rolling(12).std()
df["oni_lag1"] = df["oni"].shift(1)               # ONI is a 3-month mean -> lag it one month
df["nino34_lag1"] = df["nino34_anom"].shift(1)
df["oni_chg3"] = df["oni"].diff(3).shift(1)
df["gscpi_lag1"] = df["gscpi"].shift(1)
df["gscpi_chg3"] = df["gscpi"].diff(3)

## 4. Labels: disruption within the next h months

In [ ]:
def future_max(series, h, diff=False):
    cols = [(series.shift(-k) - series) if diff else (series.shift(-k) / series - 1) for k in range(1, h + 1)]
    out = pd.concat(cols, axis=1).max(axis=1)
    out[series.shift(-h).isna()] = np.nan       # no label where the future is unknown
    return out

RISE, SPIKE = 0.5, 0.10    # 0.5 SD rise in GSCPI ; 10% Brent rise (test other values later)
for h in (1, 2, 3):
    df[f"y_gscpi_h{h}"] = (future_max(df["gscpi"], h, diff=True) >= RISE).astype(float).where(future_max(df["gscpi"], h, diff=True).notna())
    df[f"y_brent_h{h}"] = (future_max(df["brent"], h) >= SPIKE).astype(float).where(future_max(df["brent"], h).notna())

## 5. Time-based split

In [ ]:
def split(m):
    if m <= pd.Period("2018-12"): return "train"
    if m <= pd.Period("2021-12"): return "valid"
    if m <= pd.Period("2025-12"): return "test"
    return "demo_2026"
df["split"] = df["month"].map(split)
print(df.groupby("split").size())
print(df.groupby("split")[[c for c in df if c.startswith("y_")]].mean().round(2))

split
demo_2026      9
test          48
train        252
valid         36
dtype: int64
           y_gscpi_h1  y_brent_h1  y_gscpi_h2  y_brent_h2  y_gscpi_h3  \
split                                                                   
demo_2026        0.12        0.38        0.29        0.57        0.50   
test             0.08        0.06        0.23        0.15        0.27   
train            0.05        0.13        0.12        0.32        0.19   
valid            0.25        0.17        0.42        0.39        0.53   

           y_brent_h3  
split                  
demo_2026        0.67  
test             0.23  
train            0.42  
valid            0.56  


## 6. Baseline models (statistical and machine learning)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

FEATS = ["oni_lag1", "oni_chg3", "nino34_lag1", "brent_ret1", "brent_ret3", "brent_z12",
         "gscpi_lag1", "gscpi_chg3"]
models = {
    "logistic": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                              LogisticRegression(max_iter=1000, class_weight="balanced")),
    "grad_boost": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                                 class_weight="balanced", random_state=0),
}
rows = []
for target in ("y_gscpi", "y_brent"):
    for h in (1, 2, 3):
        y = f"{target}_h{h}"
        tr_ = df[(df.split == "train") & df[y].notna()]
        for name, m in models.items():
            m.fit(tr_[FEATS], tr_[y])
            for sp in ("valid", "test"):
                d = df[(df.split == sp) & df[y].notna()]
                if d[y].nunique() < 2: continue
                p = m.predict_proba(d[FEATS])[:, 1]
                rows.append(dict(target=y, model=name, split=sp, n=len(d), pos_rate=round(d[y].mean(), 2),
                                 auc=round(roc_auc_score(d[y], p), 3),
                                 avg_precision=round(average_precision_score(d[y], p), 3)))
results = pd.DataFrame(rows)
print(results.to_string(index=False))

    target      model split  n  pos_rate   auc  avg_precision
y_gscpi_h1   logistic valid 36      0.25 0.514          0.455
y_gscpi_h1   logistic  test 48      0.08 0.608          0.170
y_gscpi_h1 grad_boost valid 36      0.25 0.560          0.318
y_gscpi_h1 grad_boost  test 48      0.08 0.767          0.402
y_gscpi_h2   logistic valid 36      0.42 0.521          0.563
y_gscpi_h2   logistic  test 48      0.23 0.656          0.360
y_gscpi_h2 grad_boost valid 36      0.42 0.410          0.378
y_gscpi_h2 grad_boost  test 48      0.23 0.813          0.696
y_gscpi_h3   logistic valid 36      0.53 0.644          0.701
y_gscpi_h3   logistic  test 48      0.27 0.705          0.442
y_gscpi_h3 grad_boost valid 36      0.53 0.461          0.589
y_gscpi_h3 grad_boost  test 48      0.27 0.820          0.677
y_brent_h1   logistic valid 36      0.17 0.233          0.125
y_brent_h1   logistic  test 48      0.06 0.289          0.083
y_brent_h1 grad_boost valid 36      0.17 0.489          0.234
y_brent_

## 7. Save outputs to Drive

In [ ]:
df.to_csv(f'{BASE}/monthly_panel.csv', index=False)
results.to_csv(f'{BASE}/baseline_results.csv', index=False)